# 19.1 机理可解释性 (Mechanistic Interpretability)

> 🕐 预估学习时间：45分钟

机理可解释性试图打开大模型的"黑盒"，定位具体电路、特征与因果通路，是安全审计、对齐验证与模型调试的关键能力。代表工作：Anthropic Circuits、Sparse Autoencoders (SAE)、Activation Patching。

本节涵盖：
- logit 归因与直接效应
- Activation Patching（激活修补）
- Sparse Autoencoder 特征分解
- Attention Head 专项分析
- 产业应用场景


## 1. Logit 归因：谁把答案推向了正确方向？

**核心思想**：把最终 logits 的变化分解到残差流中各组件（注意力头、MLP）的贡献。

**基本原理**：
- Transformer 残差流可写为 `x = embed + Σ attn + Σ mlp`
- 对目标 token 的 logit，可用线性探针 / 解嵌入矩阵 `W_U` 做投影归因
- 正贡献组件"支持"该答案，负贡献组件"反对"

**用途**：快速定位"是哪个层/头在驱动错误答案"。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)


class TinyTransformerBlock(nn.Module):
    def __init__(self, d=64, n_heads=4):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_heads, batch_first=True)
        self.ln2 = nn.LayerNorm(d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, x, return_parts=False):
        h = self.ln1(x)
        attn_out, _ = self.attn(h, h, h, need_weights=False)
        x = x + attn_out
        mlp_out = self.mlp(self.ln2(x))
        x = x + mlp_out
        if return_parts:
            return x, attn_out, mlp_out
        return x


class TinyLM(nn.Module):
    def __init__(self, vocab=100, d=64, n_layers=3):
        super().__init__()
        self.embed = nn.Embedding(vocab, d)
        self.blocks = nn.ModuleList([TinyTransformerBlock(d) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d)
        self.unembed = nn.Linear(d, vocab, bias=False)

    def forward_with_parts(self, ids):
        x = self.embed(ids)
        parts = []
        for i, block in enumerate(self.blocks):
            x, attn_out, mlp_out = block(x, return_parts=True)
            parts.append((f'L{i}.attn', attn_out),)
            parts.append((f'L{i}.mlp', mlp_out),)
        x = self.ln_f(x)
        logits = self.unembed(x)
        return logits, parts, x


model = TinyLM()
ids = torch.randint(0, 100, (2, 16))
target_token = 7
logits, parts, final_h = model.forward_with_parts(ids)

# Attribute final-position residual components to target logit via unembed
W_U = model.unembed.weight  # (vocab, d)
target_dir = W_U[target_token]  # (d,)

print('=== Logit Attribution ===')
print(f'Input ids: {ids.shape}, target token={target_token}')
contribs = []
for name, part in parts:
    # contribution of this residual write at last position
    write = part[:, -1, :]  # (B, d)
    score = (write * target_dir).sum(dim=-1).mean().item()
    contribs.append((name, score))

contribs_sorted = sorted(contribs, key=lambda x: abs(x[1]), reverse=True)
print(f'{"Component":<10} {"Mean contrib to logit":>22}')
for name, score in contribs_sorted:
    print(f'{name:<10} {score:>22.4f}')

print(f'\nKey: Logit attribution decomposes the final answer into residual-stream writes.')
print(f'Large positive/negative components are the first places to inspect for bugs or bias.')


## 2. Activation Patching：因果干预定位关键通路

**核心思想**：用"干净运行"的激活替换"被污染运行"中的对应激活，观察输出是否恢复。

**步骤**：
1. Clean run：正确提示 → 正确答案
2. Corrupt run：改写关键实体 → 错误答案
3. Patch：把 clean 的某层激活塞回 corrupt，看正确 logit 是否回升

若 patch 某组件后性能恢复，说明该组件对任务**因果必要**。


In [ ]:
def run_cached(model, ids):
    '''Forward while caching residual stream after each block.'''
    caches = []
    x = model.embed(ids)
    for block in model.blocks:
        x = block(x)
        caches.append(x.detach().clone())
    x = model.ln_f(x)
    logits = model.unembed(x)
    return logits, caches


def patched_forward(model, corrupt_ids, clean_caches, patch_layer):
    x = model.embed(corrupt_ids)
    for i, block in enumerate(model.blocks):
        x = block(x)
        if i == patch_layer:
            x = clean_caches[i]
    x = model.ln_f(x)
    return model.unembed(x)


clean_ids = torch.randint(0, 100, (4, 12))
corrupt_ids = clean_ids.clone()
corrupt_ids[:, 3] = (corrupt_ids[:, 3] + 17) % 100  # corrupt a key token

clean_logits, clean_caches = run_cached(model, clean_ids)
corrupt_logits, _ = run_cached(model, corrupt_ids)

clean_score = clean_logits[:, -1, target_token].mean().item()
corrupt_score = corrupt_logits[:, -1, target_token].mean().item()

print('=== Activation Patching ===')
print(f'Clean logit(target):   {clean_score:.4f}')
print(f'Corrupt logit(target): {corrupt_score:.4f}')
print(f'Gap: {clean_score - corrupt_score:.4f}')

print(f'\n{"Layer":>5} {"Patched logit":>14} {"Recovery%":>10}')
for layer in range(len(model.blocks)):
    patched_logits = patched_forward(model, corrupt_ids, clean_caches, layer)
    score = patched_logits[:, -1, target_token].mean().item()
    recovery = (score - corrupt_score) / (clean_score - corrupt_score + 1e-8) * 100
    print(f'{layer:>5} {score:>14.4f} {recovery:>9.1f}%')

print(f'\nKey: Activation patching finds causally necessary layers/components.')
print(f'High recovery% means that layer carries information sufficient to restore the clean behavior.')


## 3. Sparse Autoencoder：把叠加特征拆开

**问题**：神经网络表征高度叠加（superposition），单个神经元往往混合多个概念。

**SAE 思路**：
- 用更大、更稀疏的字典重建激活：`x ≈ W_dec · ReLU(W_enc · x + b)`
- L1 / Top-K 稀疏惩罚迫使每个字典单元对应更"单义"的特征
- 可用于找"拒绝有害请求"、"引用代码语法"等可解释特征

Anthropic / OpenAI 的大规模 SAE 是当前机理解释的主流工具。


In [ ]:
class SparseAutoencoder(nn.Module):
    def __init__(self, d_model=64, d_dict=256, l1_coef=1e-3):
        super().__init__()
        self.enc = nn.Linear(d_model, d_dict)
        self.dec = nn.Linear(d_dict, d_model, bias=False)
        self.l1_coef = l1_coef
        # unit-norm decoder columns improve feature interpretability
        with torch.no_grad():
            self.dec.weight.div_(self.dec.weight.norm(dim=0, keepdim=True) + 1e-8)

    def forward(self, x):
        z = F.relu(self.enc(x))
        recon = self.dec(z)
        recon_loss = F.mse_loss(recon, x)
        sparse_loss = z.abs().mean()
        loss = recon_loss + self.l1_coef * sparse_loss
        return loss, recon_loss.detach(), sparse_loss.detach(), z


# Collect residual activations as "dataset"
with torch.no_grad():
    acts = []
    for _ in range(20):
        batch = torch.randint(0, 100, (8, 16))
        _, _, h = model.forward_with_parts(batch)
        acts.append(h.reshape(-1, h.size(-1)))
    acts = torch.cat(acts, dim=0)

sae = SparseAutoencoder(d_model=64, d_dict=256, l1_coef=5e-3)
opt = torch.optim.Adam(sae.parameters(), lr=1e-2)

print('=== Sparse Autoencoder Training ===')
for step in range(80):
    idx = torch.randint(0, acts.size(0), (256,))
    loss, recon, sparse, z = sae(acts[idx])
    opt.zero_grad()
    loss.backward()
    opt.step()
    with torch.no_grad():
        sae.dec.weight.div_(sae.dec.weight.norm(dim=0, keepdim=True) + 1e-8)
    if step % 20 == 0 or step == 79:
        active = (z > 0).float().mean().item()
        print(f'step={step:02d} loss={loss.item():.4f} recon={recon.item():.4f} '
              f'l1={sparse.item():.4f} active_frac={active:.3f}')

with torch.no_grad():
    _, _, _, z_all = sae(acts[:512])
    usage = (z_all > 0).float().mean(dim=0)
    topk = usage.topk(5)

print(f'\nTop-5 most used features: idx={topk.indices.tolist()}, usage={topk.values.tolist()}')
print(f'\nKey: SAE expands and sparsifies activations into more monosemantic features.')
print(f'Decoder-column unit norm + L1 sparsity are the two practical ingredients.')


## 4. Attention Head 分析与产业用途

| 分析对象 | 方法 | 典型发现 |
|---------|------|---------|
| Induction heads | 前缀匹配复制 | few-shot / ICL 的机制基础 |
| Previous-token heads | 位置偏移注意力 | 构成 induction 电路 |
| 抑制头 | 负 logit 归因 | 参与拒绝/校准 |

**产业用途**：
- 安全：定位越狱相关电路，做定向消融
- 对齐审计：验证"诚实"特征是否被真实使用
- 调试：找出模型记住污染基准的通路
- 编辑：结合 ROME/MEMIT 做更可解释的知识更新


In [ ]:
def attention_pattern_stats(attn_module, x):
    '''Compute average attention entropy and diagonal mass.'''
    B, T, D = x.shape
    # reuse MultiheadAttention weights
    h = attn_module.ln1(x) if hasattr(attn_module, 'ln1') else x
    # manual qkv for one block
    block = attn_module
    q = k = v = block.ln1(x)
    # Use the module's in_proj if available via mha
    mha = block.attn
    attn_out, weights = mha(q, k, v, need_weights=True, average_attn_weights=True)
    # weights: (B, T, T)
    ent = -(weights * (weights + 1e-9).log()).sum(dim=-1).mean().item()
    diag = weights.diagonal(dim1=-2, dim2=-1).mean().item()
    return attn_out, ent, diag


print('=== Attention Head Diagnostics ===')
x0 = model.embed(ids)
for i, block in enumerate(model.blocks):
    _, ent, diag = attention_pattern_stats(block, x0)
    x0 = block(x0)
    print(f'Layer {i}: attn_entropy={ent:.3f}, diagonal_mass={diag:.3f}')

print(f'\nKey: Low entropy + off-diagonal structure often indicates specialized heads')
print(f'(e.g., induction). High diagonal mass suggests local/previous-token behavior.')
print(f'Mechanistic tools turn qualitative hunches into measurable interventions.')


## 课后思考题

1. Logit 归因与 Activation Patching 都能指出“重要组件”，二者的因果强度有何不同？
2. 为什么叠加（superposition）会让单神经元解释失效？SAE 如何缓解？
3. 若某安全相关特征在 SAE 中被找到，如何在生产中用于监控或干预？
4. 机理可解释性在红队、对齐审计、知识编辑中分别能解决什么问题？

---
> 本节涵盖了19.1 机理可解释性的核心概念与代码实现。建议结合实际项目需求，选择合适的技术方案，并通过实验验证不同方法的效果差异。
